In [ ]:
import math
import json
import random
import multiprocessing as mp
from pathlib import Path
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DATA_JSON = "/kaggle/input/nlp-capstone-project-gru/train_data1.json"
VAL_JSON = "/kaggle/input/nlp-capstone-project-gru/val_data1.json" 
SAVE_PATH = "/kaggle/working/transformer_en_hi_from_scratch.pth"
OUTPUT_JSON = "/kaggle/working/en_hi_val_translations.json"

BATCH_SIZE = 16               
ACCUM_STEPS = 2
MAX_LEN = 100                 
EMB_SIZE = 256
NHEAD = 8
FFN_HID_DIM = 512
NUM_ENCODER_LAYERS = 4
NUM_DECODER_LAYERS = 4
DROPOUT = 0.1
LR = 1e-4
N_EPOCHS = 15
CLIP_NORM = 1.0
NUM_WORKERS = 0              

PAD = "<pad>"
SOS = "<sos>"
EOS = "<eos>"
UNK = "<unk>"


def tokenize(s: str) -> List[str]:
    return s.strip().split()

def build_vocabs(pairs):
    src_words = set()
    tgt_words = set()
    for s, t in pairs:
        for w in tokenize(s.lower()):
            src_words.add(w)
        for w in tokenize(t.lower()):
            tgt_words.add(w)
    src_vocab_list = [PAD, SOS, EOS, UNK] + sorted(src_words)
    tgt_vocab_list = [PAD, SOS, EOS, UNK] + sorted(tgt_words)
    src_w2i = {w: i for i, w in enumerate(src_vocab_list)}
    src_i2w = {i: w for w, i in src_w2i.items()}
    tgt_w2i = {w: i for i, w in enumerate(tgt_vocab_list)}
    tgt_i2w = {i: w for w, i in tgt_w2i.items()}
    return src_vocab_list, tgt_vocab_list, src_w2i, src_i2w, tgt_w2i, tgt_i2w



class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, pairs, src_w2i, tgt_w2i, max_len=MAX_LEN):
        self.pairs = pairs
        self.src_w2i = src_w2i
        self.tgt_w2i = tgt_w2i
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        s, t = self.pairs[idx]
        src_tokens = tokenize(s.lower())[: self.max_len]
        tgt_tokens = tokenize(t.lower())[: (self.max_len - 2)]
        src_inds = [self.src_w2i.get(w, self.src_w2i[UNK]) for w in src_tokens]
        tgt_inds = [self.tgt_w2i[SOS]] + [self.tgt_w2i.get(w, self.tgt_w2i[UNK]) for w in tgt_tokens] + [self.tgt_w2i[EOS]]
        return torch.tensor(src_inds, dtype=torch.long), torch.tensor(tgt_inds, dtype=torch.long)

def collate_fn_cpu(batch):
    src_batch, tgt_batch = zip(*batch)
    src_padded = nn.utils.rnn.pad_sequence(src_batch, padding_value=src_w2i[PAD], batch_first=True)
    tgt_padded = nn.utils.rnn.pad_sequence(tgt_batch, padding_value=tgt_w2i[PAD], batch_first=True)
    return src_padded, tgt_padded



class PositionalEncoding(nn.Module):
    def __init__(self, emb_size: int, dropout: float = 0.1, maxlen: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(maxlen, emb_size)
        position = torch.arange(0, maxlen, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, emb_size, 2).float() * (-math.log(10000.0) / emb_size))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor):
        x = x + self.pe[:, : x.size(1), :].to(x.device)
        return self.dropout(x)




class Seq2SeqTransformer(nn.Module):
    def __init__(self, num_encoder_layers: int, num_decoder_layers: int, emb_size: int, nhead: int,
                 src_vocab_size: int, tgt_vocab_size: int, dim_feedforward: int = 512, dropout: float = 0.1):
        super().__init__()
        self.transformer = nn.Transformer(d_model=emb_size,
                                          nhead=nhead,
                                          num_encoder_layers=num_encoder_layers,
                                          num_decoder_layers=num_decoder_layers,
                                          dim_feedforward=dim_feedforward,
                                          dropout=dropout,
                                          batch_first=True)
        self.generator = nn.Linear(emb_size, tgt_vocab_size)
        self.src_tok_emb = nn.Embedding(src_vocab_size, emb_size, padding_idx=src_w2i[PAD])
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, emb_size, padding_idx=tgt_w2i[PAD])
        self.positional_encoding = PositionalEncoding(emb_size, dropout=dropout)

    def forward(self, src, tgt_in, tgt_mask, src_key_padding_mask, tgt_key_padding_mask, memory_key_padding_mask):
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt_in))
        memory = self.transformer.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)
        outs = self.transformer.decoder(tgt_emb, memory, tgt_mask=tgt_mask,
                                        tgt_key_padding_mask=tgt_key_padding_mask,
                                        memory_key_padding_mask=memory_key_padding_mask)
        logits = self.generator(outs)
        return logits

    def encode(self, src, src_key_padding_mask):
        return self.transformer.encoder(self.positional_encoding(self.src_tok_emb(src)), src_key_padding_mask=src_key_padding_mask)

    def decode(self, tgt, memory, tgt_mask, tgt_key_padding_mask, memory_key_padding_mask):
        return self.transformer.decoder(self.positional_encoding(self.tgt_tok_emb(tgt)), memory, tgt_mask=tgt_mask,
                                        tgt_key_padding_mask=tgt_key_padding_mask, memory_key_padding_mask=memory_key_padding_mask)




def generate_square_subsequent_mask(sz: int) -> torch.Tensor:
    mask = (torch.triu(torch.ones((sz, sz), device=DEVICE)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float("-inf")).masked_fill(mask == 1, float(0.0))
    return mask

def create_padding_mask(seq: torch.Tensor, pad_idx: int):
    return (seq == pad_idx)


with open(DATA_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

train_dict = None
if isinstance(data, dict):
    for candidate in ["English-Hindi", "English-Hindi", "Train", "train"]:
        if candidate in data:
            train_dict = data[candidate]
            break
    if train_dict is None and all(isinstance(v, dict) and "source" in v and "target" in v for v in data.values()):
        train_dict = data

if train_dict is None:
    raise ValueError("Cannot find training data in JSON. Check DATA_JSON structure.")

if isinstance(train_dict, dict) and "Train" in train_dict:
    items = train_dict["Train"]
else:
    items = train_dict

pairs = [(v["source"], v["target"]) for k, v in items.items()]
random.shuffle(pairs)
split_idx = int(0.95 * len(pairs))
train_pairs = pairs[:split_idx]
val_pairs = pairs[split_idx:]

print(f"Train pairs: {len(train_pairs)}, Val pairs: {len(val_pairs)}")

src_vocab_list, tgt_vocab_list, src_w2i, src_i2w, tgt_w2i, tgt_i2w = build_vocabs(train_pairs)
SRC_VOCAB_SIZE = len(src_vocab_list)
TGT_VOCAB_SIZE = len(tgt_vocab_list)
print("Vocab sizes -> src:", SRC_VOCAB_SIZE, "tgt:", TGT_VOCAB_SIZE)


train_dataset = TranslationDataset(train_pairs, src_w2i, tgt_w2i, max_len=MAX_LEN)
val_dataset = TranslationDataset(val_pairs, src_w2i, tgt_w2i, max_len=MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn_cpu,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn_cpu,
                        num_workers=NUM_WORKERS, pin_memory=True)


model = Seq2SeqTransformer(NUM_ENCODER_LAYERS, NUM_DECODER_LAYERS, EMB_SIZE, NHEAD, SRC_VOCAB_SIZE, TGT_VOCAB_SIZE, FFN_HID_DIM, DROPOUT).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=tgt_w2i[PAD])

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler(enabled=use_amp)


def train_one_epoch(model, optimizer, dataloader, epoch_idx):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    torch.cuda.empty_cache()
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Train Epoch {epoch_idx}")
    for step, (src_cpu, tgt_cpu) in pbar:

        src = src_cpu.to(DEVICE)
        tgt = tgt_cpu.to(DEVICE)

    
        src_key_padding_mask = create_padding_mask(src, src_w2i[PAD])
        tgt_input = tgt[:, :-1]
        tgt_out = tgt[:, 1:]
        tgt_key_padding_mask = create_padding_mask(tgt_input, tgt_w2i[PAD])
        memory_key_padding_mask = src_key_padding_mask

        tgt_mask = generate_square_subsequent_mask(tgt_input.size(1)) if tgt_input.size(1) > 0 else None

        with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu", enabled=use_amp):
            logits = model(src, tgt_input, tgt_mask, src_key_padding_mask, tgt_key_padding_mask, memory_key_padding_mask)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            loss_value = loss.item()

        scaler.scale(loss / ACCUM_STEPS).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(dataloader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss_value
        pbar.set_postfix({"avg_loss": f"{(total_loss / (step+1)):.4f}"})

    return total_loss / len(dataloader)

def evaluate(model, dataloader):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for src_cpu, tgt_cpu in tqdm(dataloader, desc="Eval"):
            src = src_cpu.to(DEVICE)
            tgt = tgt_cpu.to(DEVICE)

            src_key_padding_mask = create_padding_mask(src, src_w2i[PAD])
            tgt_input = tgt[:, :-1]
            tgt_out = tgt[:, 1:]
            tgt_key_padding_mask = create_padding_mask(tgt_input, tgt_w2i[PAD])
            tgt_mask = generate_square_subsequent_mask(tgt_input.size(1)) if tgt_input.size(1) > 0 else None

            logits = model(src, tgt_input, tgt_mask, src_key_padding_mask, tgt_key_padding_mask, src_key_padding_mask)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            total_loss += loss.item()
    return total_loss / len(dataloader)


def greedy_decode(model, src_sentence: str, max_len: int = 50):
    model.eval()
    tokens = [src_w2i.get(w, src_w2i[UNK]) for w in tokenize(src_sentence.lower())][:MAX_LEN]
    src = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(DEVICE)
    src_key_padding_mask = create_padding_mask(src, src_w2i[PAD])

    with torch.no_grad():
        memory = model.encode(src, src_key_padding_mask)
        ys = torch.tensor([[tgt_w2i[SOS]]], dtype=torch.long).to(DEVICE)
        for i in range(max_len):
            tgt_mask = generate_square_subsequent_mask(ys.size(1))
            out = model.decode(ys, memory, tgt_mask, create_padding_mask(ys, tgt_w2i[PAD]), src_key_padding_mask)
            out = out[:, -1:, :]
            prob = model.generator(out) 
            _, next_word = torch.max(prob, dim=-1)
            next_word = next_word.item()
            ys = torch.cat([ys, torch.tensor([[next_word]], dtype=torch.long).to(DEVICE)], dim=1)
            if next_word == tgt_w2i[EOS]:
                break
    decoded = [tgt_i2w[idx] for idx in ys.squeeze().tolist()[1:] if idx != tgt_w2i[EOS]]
    return " ".join(decoded)


best_val = float("inf")
for epoch in range(1, N_EPOCHS + 1):
    train_loss = train_one_epoch(model, optimizer, train_loader, epoch)
    val_loss = evaluate(model, val_loader) if len(val_loader) > 0 else float("inf")
    print(f"Epoch {epoch} -> train: {train_loss:.4f}, val: {val_loss:.4f}")
    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            "model_state_dict": model.state_dict(),
            "src_vocab_list": src_vocab_list,
            "tgt_vocab_list": tgt_vocab_list
        }, SAVE_PATH)
        print("Saved best model to", SAVE_PATH)


print("\n Sample translations")
for _ in range(5):
    if len(val_pairs) == 0:
        break
    s, t = random.choice(val_pairs)
    pred = greedy_decode(model, s, max_len=40)
    print("EN:", s)
    print("GT:", t)
    print("PRED:", pred)
    print("-" * 40)


if Path(VAL_JSON).exists():
    with open(VAL_JSON, "r", encoding="utf-8") as f:
        val_data = json.load(f)


    if "root" in val_data:
        val_data = val_data["root"]

    
    val_items = val_data.get("English-Hindi", {}).get("Validation", {})
    results = {}

    print(f"\n Translating for English-Hindi ({len(val_items)} items)...")

    for key, obj in tqdm(val_items.items(), desc="Translating English-Hindi"):
        src_text = obj.get("source", "")
        pred = greedy_decode(model, src_text, max_len=50)
        results[key] = {"source": src_text, "prediction": pred}

    out_data = {"English-Hindi": {"Validation": results}}
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(out_data, f, ensure_ascii=False, indent=4)

    print(f"\n English-Hindi translations saved to {OUTPUT_JSON}")
else:
    print(f"VAL_JSON not found at {VAL_JSON}, skipping translation.")


In [ ]:
import math
import json
import random
import multiprocessing as mp
from pathlib import Path
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DATA_JSON = "/kaggle/input/bi-lstm-english-beng/train_data1.json"
VAL_JSON = "/kaggle/input/bi-lstm-english-beng/val_data1.json" 
SAVE_PATH = "/kaggle/working/transformer_en_bn_from_scratch.pth"
OUTPUT_JSON = "/kaggle/working/en_bn_val_translations.json"

BATCH_SIZE = 16               
ACCUM_STEPS = 2               
MAX_LEN = 100                
EMB_SIZE = 256
NHEAD = 8
FFN_HID_DIM = 512
NUM_ENCODER_LAYERS = 4
NUM_DECODER_LAYERS = 4
DROPOUT = 0.1
LR = 1e-4
N_EPOCHS = 15
CLIP_NORM = 1.0
NUM_WORKERS = 0           

PAD = "<pad>"
SOS = "<sos>"
EOS = "<eos>"
UNK = "<unk>"


def tokenize(s: str) -> List[str]:
    return s.strip().split()

def build_vocabs(pairs):
    src_words = set()
    tgt_words = set()
    for s, t in pairs:
        for w in tokenize(s.lower()):
            src_words.add(w)
        for w in tokenize(t.lower()):
            tgt_words.add(w)
    src_vocab_list = [PAD, SOS, EOS, UNK] + sorted(src_words)
    tgt_vocab_list = [PAD, SOS, EOS, UNK] + sorted(tgt_words)
    src_w2i = {w: i for i, w in enumerate(src_vocab_list)}
    src_i2w = {i: w for w, i in src_w2i.items()}
    tgt_w2i = {w: i for i, w in enumerate(tgt_vocab_list)}
    tgt_i2w = {i: w for w, i in tgt_w2i.items()}
    return src_vocab_list, tgt_vocab_list, src_w2i, src_i2w, tgt_w2i, tgt_i2w


class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, pairs, src_w2i, tgt_w2i, max_len=MAX_LEN):
        self.pairs = pairs
        self.src_w2i = src_w2i
        self.tgt_w2i = tgt_w2i
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        s, t = self.pairs[idx]
        src_tokens = tokenize(s.lower())[: self.max_len]
        tgt_tokens = tokenize(t.lower())[: (self.max_len - 2)]
        src_inds = [self.src_w2i.get(w, self.src_w2i[UNK]) for w in src_tokens]
        tgt_inds = [self.tgt_w2i[SOS]] + [self.tgt_w2i.get(w, self.tgt_w2i[UNK]) for w in tgt_tokens] + [self.tgt_w2i[EOS]]
        return torch.tensor(src_inds, dtype=torch.long), torch.tensor(tgt_inds, dtype=torch.long)

def collate_fn_cpu(batch):
    src_batch, tgt_batch = zip(*batch)
    src_padded = nn.utils.rnn.pad_sequence(src_batch, padding_value=src_w2i[PAD], batch_first=True)
    tgt_padded = nn.utils.rnn.pad_sequence(tgt_batch, padding_value=tgt_w2i[PAD], batch_first=True)
    return src_padded, tgt_padded


class PositionalEncoding(nn.Module):
    def __init__(self, emb_size: int, dropout: float = 0.1, maxlen: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(maxlen, emb_size)
        position = torch.arange(0, maxlen, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, emb_size, 2).float() * (-math.log(10000.0) / emb_size))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor):
        x = x + self.pe[:, : x.size(1), :].to(x.device)
        return self.dropout(x)


class Seq2SeqTransformer(nn.Module):
    def __init__(self, num_encoder_layers: int, num_decoder_layers: int, emb_size: int, nhead: int,
                 src_vocab_size: int, tgt_vocab_size: int, dim_feedforward: int = 512, dropout: float = 0.1):
        super().__init__()
        self.transformer = nn.Transformer(d_model=emb_size,
                                          nhead=nhead,
                                          num_encoder_layers=num_encoder_layers,
                                          num_decoder_layers=num_decoder_layers,
                                          dim_feedforward=dim_feedforward,
                                          dropout=dropout,
                                          batch_first=True)
        self.generator = nn.Linear(emb_size, tgt_vocab_size)
        self.src_tok_emb = nn.Embedding(src_vocab_size, emb_size, padding_idx=src_w2i[PAD])
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, emb_size, padding_idx=tgt_w2i[PAD])
        self.positional_encoding = PositionalEncoding(emb_size, dropout=dropout)

    def forward(self, src, tgt_in, tgt_mask, src_key_padding_mask, tgt_key_padding_mask, memory_key_padding_mask):
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt_in))
        memory = self.transformer.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)
        outs = self.transformer.decoder(tgt_emb, memory, tgt_mask=tgt_mask,
                                        tgt_key_padding_mask=tgt_key_padding_mask,
                                        memory_key_padding_mask=memory_key_padding_mask)
        logits = self.generator(outs)
        return logits

    def encode(self, src, src_key_padding_mask):
        return self.transformer.encoder(self.positional_encoding(self.src_tok_emb(src)), src_key_padding_mask=src_key_padding_mask)

    def decode(self, tgt, memory, tgt_mask, tgt_key_padding_mask, memory_key_padding_mask):
        return self.transformer.decoder(self.positional_encoding(self.tgt_tok_emb(tgt)), memory, tgt_mask=tgt_mask,
                                        tgt_key_padding_mask=tgt_key_padding_mask, memory_key_padding_mask=memory_key_padding_mask)


def generate_square_subsequent_mask(sz: int) -> torch.Tensor:
    mask = (torch.triu(torch.ones((sz, sz), device=DEVICE)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float("-inf")).masked_fill(mask == 1, float(0.0))
    return mask

def create_padding_mask(seq: torch.Tensor, pad_idx: int):
    return (seq == pad_idx)



with open(DATA_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

train_dict = None
if isinstance(data, dict):
    for candidate in ["English-Bengali", "English-Bengali", "Train", "train"]:
        if candidate in data:
            train_dict = data[candidate]
            break
    if train_dict is None and all(isinstance(v, dict) and "source" in v and "target" in v for v in data.values()):
        train_dict = data

if train_dict is None:
    raise ValueError("Cannot find training data in JSON. Check DATA_JSON structure.")

if isinstance(train_dict, dict) and "Train" in train_dict:
    items = train_dict["Train"]
else:
    items = train_dict

pairs = [(v["source"], v["target"]) for k, v in items.items()]
random.shuffle(pairs)
split_idx = int(0.95 * len(pairs))
train_pairs = pairs[:split_idx]
val_pairs = pairs[split_idx:]

print(f"Train pairs: {len(train_pairs)}, Val pairs: {len(val_pairs)}")

src_vocab_list, tgt_vocab_list, src_w2i, src_i2w, tgt_w2i, tgt_i2w = build_vocabs(train_pairs)
SRC_VOCAB_SIZE = len(src_vocab_list)
TGT_VOCAB_SIZE = len(tgt_vocab_list)
print("Vocab sizes src:", SRC_VOCAB_SIZE, "tgt:", TGT_VOCAB_SIZE)


train_dataset = TranslationDataset(train_pairs, src_w2i, tgt_w2i, max_len=MAX_LEN)
val_dataset = TranslationDataset(val_pairs, src_w2i, tgt_w2i, max_len=MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn_cpu,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn_cpu,
                        num_workers=NUM_WORKERS, pin_memory=True)



model = Seq2SeqTransformer(NUM_ENCODER_LAYERS, NUM_DECODER_LAYERS, EMB_SIZE, NHEAD, SRC_VOCAB_SIZE, TGT_VOCAB_SIZE, FFN_HID_DIM, DROPOUT).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=tgt_w2i[PAD])

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler(enabled=use_amp)


def train_one_epoch(model, optimizer, dataloader, epoch_idx):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    torch.cuda.empty_cache()
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Train Epoch {epoch_idx}")
    for step, (src_cpu, tgt_cpu) in pbar:
        src = src_cpu.to(DEVICE)
        tgt = tgt_cpu.to(DEVICE)

        src_key_padding_mask = create_padding_mask(src, src_w2i[PAD])
        tgt_input = tgt[:, :-1]
        tgt_out = tgt[:, 1:]
        tgt_key_padding_mask = create_padding_mask(tgt_input, tgt_w2i[PAD])
        memory_key_padding_mask = src_key_padding_mask

        tgt_mask = generate_square_subsequent_mask(tgt_input.size(1)) if tgt_input.size(1) > 0 else None

        with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu", enabled=use_amp):
            logits = model(src, tgt_input, tgt_mask, src_key_padding_mask, tgt_key_padding_mask, memory_key_padding_mask)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            loss_value = loss.item()

        scaler.scale(loss / ACCUM_STEPS).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(dataloader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss_value
        pbar.set_postfix({"avg_loss": f"{(total_loss / (step+1)):.4f}"})

    return total_loss / len(dataloader)

def evaluate(model, dataloader):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for src_cpu, tgt_cpu in tqdm(dataloader, desc="Eval"):
            src = src_cpu.to(DEVICE)
            tgt = tgt_cpu.to(DEVICE)

            src_key_padding_mask = create_padding_mask(src, src_w2i[PAD])
            tgt_input = tgt[:, :-1]
            tgt_out = tgt[:, 1:]
            tgt_key_padding_mask = create_padding_mask(tgt_input, tgt_w2i[PAD])
            tgt_mask = generate_square_subsequent_mask(tgt_input.size(1)) if tgt_input.size(1) > 0 else None

            logits = model(src, tgt_input, tgt_mask, src_key_padding_mask, tgt_key_padding_mask, src_key_padding_mask)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            total_loss += loss.item()
    return total_loss / len(dataloader)


def greedy_decode(model, src_sentence: str, max_len: int = 50):
    model.eval()
    tokens = [src_w2i.get(w, src_w2i[UNK]) for w in tokenize(src_sentence.lower())][:MAX_LEN]
    src = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(DEVICE)
    src_key_padding_mask = create_padding_mask(src, src_w2i[PAD])

    with torch.no_grad():
        memory = model.encode(src, src_key_padding_mask)
        ys = torch.tensor([[tgt_w2i[SOS]]], dtype=torch.long).to(DEVICE)
        for i in range(max_len):
            tgt_mask = generate_square_subsequent_mask(ys.size(1))
            out = model.decode(ys, memory, tgt_mask, create_padding_mask(ys, tgt_w2i[PAD]), src_key_padding_mask)
            out = out[:, -1:, :]
            prob = model.generator(out)  
            _, next_word = torch.max(prob, dim=-1)
            next_word = next_word.item()
            ys = torch.cat([ys, torch.tensor([[next_word]], dtype=torch.long).to(DEVICE)], dim=1)
            if next_word == tgt_w2i[EOS]:
                break
    decoded = [tgt_i2w[idx] for idx in ys.squeeze().tolist()[1:] if idx != tgt_w2i[EOS]]
    return " ".join(decoded)


best_val = float("inf")
for epoch in range(1, N_EPOCHS + 1):
    train_loss = train_one_epoch(model, optimizer, train_loader, epoch)
    val_loss = evaluate(model, val_loader) if len(val_loader) > 0 else float("inf")
    print(f"Epoch {epoch} -> train: {train_loss:.4f}, val: {val_loss:.4f}")
    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            "model_state_dict": model.state_dict(),
            "src_vocab_list": src_vocab_list,
            "tgt_vocab_list": tgt_vocab_list
        }, SAVE_PATH)
        print("Saved best model to", SAVE_PATH)


print("\n Sample translations ")
for _ in range(5):
    if len(val_pairs) == 0:
        break
    s, t = random.choice(val_pairs)
    pred = greedy_decode(model, s, max_len=40)
    print("EN:", s)
    print("GT:", t)
    print("PRED:", pred)
    print("-" * 40)


if Path(VAL_JSON).exists():
    with open(VAL_JSON, "r", encoding="utf-8") as f:
        val_data = json.load(f)

    if "root" in val_data:
        val_data = val_data["root"]

    val_items = val_data.get("English-Bengali", {}).get("Validation", {})
    results = {}

    print(f"\n Translating for English-Bengali ({len(val_items)} items)...")

    for key, obj in tqdm(val_items.items(), desc="Translating English-Bengali"):
        src_text = obj.get("source", "")
        pred = greedy_decode(model, src_text, max_len=50)
        results[key] = {"source": src_text, "prediction": pred}

    out_data = {"English-Bengali": {"Validation": results}}
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(out_data, f, ensure_ascii=False, indent=4)

    print(f"\nEnglish-Bengali translations saved to {OUTPUT_JSON}")
else:
    print(f"VAL_JSON not found at {VAL_JSON}, skipping translation.")






<h1>Inference using Beam Search </h1>

In [ ]:
MODEL_PATH = "/kaggle/input/last-supper-eng-bengali-transformer/pytorch/default/1/transformer_en_bn_from_scratch.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"
MAX_LEN = 50


def tokenize(s):
    return s.strip().split()


class PositionalEncoding(nn.Module):
    def __init__(self, emb_size: int, dropout: float = 0.1, maxlen: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(maxlen, emb_size)
        position = torch.arange(0, maxlen, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, emb_size, 2).float() * (-math.log(10000.0) / emb_size))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)
    def forward(self, x):
        x = x + self.pe[:, : x.size(1), :].to(x.device)
        return self.dropout(x)


class Seq2SeqTransformer(nn.Module):
    def __init__(self, num_encoder_layers, num_decoder_layers, emb_size, nhead,
                 src_vocab_size, tgt_vocab_size, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.transformer = nn.Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.generator = nn.Linear(emb_size, tgt_vocab_size)
        self.src_tok_emb = nn.Embedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, emb_size)
        self.positional_encoding = PositionalEncoding(emb_size, dropout)

    def encode(self, src, src_key_padding_mask):
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        return self.transformer.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)

    def decode(self, tgt, memory, tgt_mask, tgt_key_padding_mask, memory_key_padding_mask):
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt))
        return self.transformer.decoder(
            tgt_emb, memory, tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )

    def forward(self, src, tgt_in, tgt_mask, src_key_padding_mask, tgt_key_padding_mask, memory_key_padding_mask):
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt_in))
        outs = self.transformer(
            src_emb, tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )
        return self.generator(outs)



def generate_square_subsequent_mask(sz):
    mask = (torch.triu(torch.ones((sz, sz), device=DEVICE)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, 0.0)
    return mask

def create_padding_mask(seq, pad_idx):
    return (seq == pad_idx)



checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
src_vocab_list = checkpoint["src_vocab_list"]
tgt_vocab_list = checkpoint["tgt_vocab_list"]

src_w2i = {w: i for i, w in enumerate(src_vocab_list)}
tgt_w2i = {w: i for i, w in enumerate(tgt_vocab_list)}
tgt_i2w = {i: w for w, i in tgt_w2i.items()}

model = Seq2SeqTransformer(
    num_encoder_layers=4, 
    num_decoder_layers=4,
    emb_size=256,
    nhead=8,
    src_vocab_size=len(src_w2i),
    tgt_vocab_size=len(tgt_w2i),
    dim_feedforward=512,
    dropout=0.1
).to(DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model and vocab loaded successfully!")



def beam_search_decode(model, src_sentence, beam_size=5, max_len=50):
    model.eval()
    tokens = [src_w2i.get(w, src_w2i[UNK]) for w in tokenize(src_sentence.lower())][:max_len]
    src = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(DEVICE)
    src_mask = create_padding_mask(src, src_w2i[PAD])
    memory = model.encode(src, src_mask)

    beams = [(torch.tensor([[tgt_w2i[SOS]]], device=DEVICE), 0.0)]
    completed = []

    for _ in range(max_len):
        new_beams = []
        for seq, score in beams:
            if seq[0, -1].item() == tgt_w2i[EOS]:
                completed.append((seq, score))
                continue

            tgt_mask = generate_square_subsequent_mask(seq.size(1))
            out = model.decode(seq, memory, tgt_mask, create_padding_mask(seq, tgt_w2i[PAD]), src_mask)
            logits = model.generator(out[:, -1])
            probs = torch.log_softmax(logits, dim=-1)
            topk_probs, topk_idx = probs.topk(beam_size)

            for prob, idx in zip(topk_probs[0], topk_idx[0]):
                new_seq = torch.cat([seq, idx.view(1, 1)], dim=1)
                new_beams.append((new_seq, score + prob.item()))

        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]

        if all(seq[0, -1].item() == tgt_w2i[EOS] for seq, _ in beams):
            break

    completed = completed or beams
    best_seq, best_score = max(completed, key=lambda x: x[1])
    decoded = [tgt_i2w[i.item()] for i in best_seq.squeeze()[1:] if i.item() != tgt_w2i[EOS]]
    return " ".join(decoded)



VAL_JSON = "/kaggle/input/bi-lstm-english-beng/val_data1.json"
OUTPUT_JSON = "/kaggle/working/last-supper_en_bn_val_beam_search_translations.json"


if Path(VAL_JSON).exists():
    with open(VAL_JSON, "r", encoding="utf-8") as f:
        val_data = json.load(f)

    if "root" in val_data:
        val_data = val_data["root"]

    val_items = val_data.get("English-Bengali", {}).get("Validation", {})
    # val_items = dict(list(val_items.items())[:5]) 


    results = {}

    print(f"\nTranslating for English-Bengali ({len(val_items)} items)...")

    for key, obj in tqdm(val_items.items(), desc="Translating English-Bengali"):
        src_text = obj.get("source", "")
        pred = beam_search_decode(model, src_text, beam_size=5)
        results[key] = {"source": src_text, "prediction": pred}

    out_data = {"English-Bengali": {"Validation": results}}
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(out_data, f, ensure_ascii=False, indent=4)

    print(f"\nEnglish-Bengali translations saved to {OUTPUT_JSON}")
else:
    print(f"VAL_JSON not found at {VAL_JSON}, skipping translation.")





In [ ]:
MODEL_PATH = "/kaggle/input/last-ride-50epoch/pytorch/default/1/transformer_en_hi_from_scratch_50.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"
MAX_LEN = 50


def tokenize(s):
    return s.strip().split()

class PositionalEncoding(nn.Module):
    def __init__(self, emb_size: int, dropout: float = 0.1, maxlen: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(maxlen, emb_size)
        position = torch.arange(0, maxlen, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, emb_size, 2).float() * (-math.log(10000.0) / emb_size))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)
    def forward(self, x):
        x = x + self.pe[:, : x.size(1), :].to(x.device)
        return self.dropout(x)



class Seq2SeqTransformer(nn.Module):
    def __init__(self, num_encoder_layers, num_decoder_layers, emb_size, nhead,
                 src_vocab_size, tgt_vocab_size, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.transformer = nn.Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.generator = nn.Linear(emb_size, tgt_vocab_size)
        self.src_tok_emb = nn.Embedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, emb_size)
        self.positional_encoding = PositionalEncoding(emb_size, dropout)

    def encode(self, src, src_key_padding_mask):
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        return self.transformer.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)

    def decode(self, tgt, memory, tgt_mask, tgt_key_padding_mask, memory_key_padding_mask):
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt))
        return self.transformer.decoder(
            tgt_emb, memory, tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )

    def forward(self, src, tgt_in, tgt_mask, src_key_padding_mask, tgt_key_padding_mask, memory_key_padding_mask):
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt_in))
        outs = self.transformer(
            src_emb, tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )
        return self.generator(outs)



def generate_square_subsequent_mask(sz):
    mask = (torch.triu(torch.ones((sz, sz), device=DEVICE)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, 0.0)
    return mask

def create_padding_mask(seq, pad_idx):
    return (seq == pad_idx)



checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
src_vocab_list = checkpoint["src_vocab_list"]
tgt_vocab_list = checkpoint["tgt_vocab_list"]

src_w2i = {w: i for i, w in enumerate(src_vocab_list)}
tgt_w2i = {w: i for i, w in enumerate(tgt_vocab_list)}
tgt_i2w = {i: w for w, i in tgt_w2i.items()}

model = Seq2SeqTransformer(
    num_encoder_layers=4,  
    num_decoder_layers=4,
    emb_size=256,
    nhead=8,
    src_vocab_size=len(src_w2i),
    tgt_vocab_size=len(tgt_w2i),
    dim_feedforward=512,
    dropout=0.1
).to(DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model and vocab loaded successfully!")


def beam_search_decode(model, src_sentence, beam_size=5, max_len=50):
    model.eval()
    tokens = [src_w2i.get(w, src_w2i[UNK]) for w in tokenize(src_sentence.lower())][:max_len]
    src = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(DEVICE)
    src_mask = create_padding_mask(src, src_w2i[PAD])
    memory = model.encode(src, src_mask)

    beams = [(torch.tensor([[tgt_w2i[SOS]]], device=DEVICE), 0.0)]
    completed = []

    for _ in range(max_len):
        new_beams = []
        for seq, score in beams:
            if seq[0, -1].item() == tgt_w2i[EOS]:
                completed.append((seq, score))
                continue

            tgt_mask = generate_square_subsequent_mask(seq.size(1))
            out = model.decode(seq, memory, tgt_mask, create_padding_mask(seq, tgt_w2i[PAD]), src_mask)
            logits = model.generator(out[:, -1])
            probs = torch.log_softmax(logits, dim=-1)
            topk_probs, topk_idx = probs.topk(beam_size)

            for prob, idx in zip(topk_probs[0], topk_idx[0]):
                new_seq = torch.cat([seq, idx.view(1, 1)], dim=1)
                new_beams.append((new_seq, score + prob.item()))

        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]

        if all(seq[0, -1].item() == tgt_w2i[EOS] for seq, _ in beams):
            break

    completed = completed or beams
    best_seq, best_score = max(completed, key=lambda x: x[1])
    decoded = [tgt_i2w[i.item()] for i in best_seq.squeeze()[1:] if i.item() != tgt_w2i[EOS]]
    return " ".join(decoded)




VAL_JSON = "/kaggle/input/validation-nlp/test_data1_final.json"  
OUTPUT_JSON = "/kaggle/working/test_en_hi_val_topk_sampling_translations50.json"


if Path(VAL_JSON).exists():
    with open(VAL_JSON, "r", encoding="utf-8") as f:
        val_data = json.load(f)

    if "root" in val_data:
        val_data = val_data["root"]

    val_items = val_data.get("English-Hindi", {}).get("Test", {})
    # val_items = dict(list(val_items.items())[:5]) 
    results = {}

    print(f"\nTranslating for English-Hindi ({len(val_items)} items)...")

    for key, obj in tqdm(val_items.items(), desc="Translating English-Hindi"):
        src_text = obj.get("source", "")
        pred = topk_sampling_decode(model, src_text, k=50)

        results[key] = {"source": src_text, "prediction": pred}

    out_data = {"English-Hindi": {"Validation": results}}
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(out_data, f, ensure_ascii=False, indent=4)

    print(f"\nEnglish-Hindi translations saved to {OUTPUT_JSON}")
else:
    print(f"VAL_JSON not found at {VAL_JSON}, skipping translation.")